# Session 01 — Statistical questions and data in Python

Revision ID: S01-r1

**Question:** What must a software team know about incident records before comparing versions?

## Learning outcomes

1. Identify units, population, available sample, parameter, and statistic.
2. Classify variables by meaning rather than storage type.
3. Load a CSV and inspect dimensions, types, and nonmissing counts.
4. Write an audit with evidence, quality concerns, and a limitation.

## How to use this notebook
Open in Jupyter or upload this `.ipynb` through Colab **File → Upload notebook**.
Run cells in order. The complete CSV is embedded for offline use; no upload is
required. To practice file loading, upload the provided CSV and replace the
StringIO source with `"software_incidents.csv"`. Keep a copy of your responses.
Allow 20 minutes for demonstration and 30 for exercises E1–E3.
Runnable placeholders do not mean the exercises are complete: replace them and
write the requested interpretations. Use Restart and Run All before submission.
Only pandas is required; Colab normally provides it. No random operations occur.

In [1]:
import pandas as pd
from io import StringIO
print("pandas", pd.__version__)

pandas 3.0.6


## Dataset provenance

Synthetic course data, offered under CC0 1.0. These are fictional closed incidents, not randomly sampled observations; they cannot support claims about real deployments.

# Data dictionary

Intended observational unit: a fictional closed software incident.
File: `software_incidents.csv`, UTF-8, comma-separated, header present.
Blank fields mean unrecorded information. Valid values describe the intended
schema; the raw export includes deliberate quality defects.

| Variable | Meaning | Statistical type | Units / valid values |
|---|---|---|---|
| `incident_id` | Incident identifier | Nominal identifier | `INC-001` through `INC-024`; unique after deduplication |
| `service` | Service affected | Nominal categorical | `api`, `web`, `worker`; may be blank |
| `severity` | Ordered operational severity | Ordinal categorical | `low` < `medium` < `high`; distances are not numeric |
| `resolution_hours` | Elapsed opening-to-closure time | Numerical, conceptually continuous | Hours, nonnegative; blank = unknown |
| `deploy_version` | Version at incident opening | Nominal categorical | `v1`, `v2`; not randomized treatment |
| `customer_impact` | Recorded customer impact | Binary categorical | `yes`, `no` |

Rounding duration to whole hours does not make it a count.
Derived in Session 2: `resolution_days = resolution_hours / 24`.
A display label `unknown` for missing service does not recover the true service.
In grouped summaries, `size` counts records; `count` counts nonmissing durations.

## Concepts before code
An observational unit is the entity described by a measurement. A population
is the full set of interest, and a sample is the subset observed. A population
parameter describes the population; a sample statistic describes the sample.
Here, the fictional target is all closed incidents in an imagined reporting
period, but the export is deliberately constructed, not a probability sample.
`DataFrame` means a labeled table; `Series` means a labeled one-dimensional column.
An ID is a label. Nominal categories have no measurement order; ordinal categories
have an order but not equal numerical steps. Duration measures elapsed time.

**Predict:** Can a column stored as an integer still be an identifier?

## Classify data before choosing a summary
Statistical classification follows the meaning of the measurement:

| Class | Example |
|---|---|
| Nominal categorical | service or incident identifier |
| Ordinal categorical | severity: low, medium, high |
| Discrete quantitative | a count of incidents |
| Continuous quantitative | resolution time in hours |

Python and pandas add a storage classification. Text labels may be stored as
`str`, `object`, or `category`; counts as `int`; measurements as `float`; flags
as `bool`; and dates as `datetime`. Storage type does not determine measurement
meaning or certify validity.

## Python syntax for common data types
```python
label = "api"                              # str
severity = pd.Categorical(["low", "high"],
                          categories=["low", "medium", "high"],
                          ordered=True)     # ordered category
count = 25                                  # int
hours = 2.5                                 # float
customer_impact = True                      # bool
created_at = pd.Timestamp("2026-09-20")    # datetime
```

Use `type(value)` for a Python value and `series.dtype` for a pandas column.
Do not convert an identifier to a number just because it contains digits.

In [2]:
demo = pd.read_csv(StringIO("incident_id,service,resolution_hours\nD1,api,2\nD2,web,5\n"))
display(demo.head())
print("Rows and columns:", demo.shape)

,incident_id,service,resolution_hours
0,D1,api,2
1,D2,web,5


Rows and columns: (2, 3)


The demonstration has two records and three variables. Whole-hour recording does not turn elapsed time into a count. A mean of these two durations would be a statistic of these records. **Explain:** What additional information would justify treating these rows as distinct incidents?

## Load the course export

Predict what `head()` can reveal and what it cannot establish about the rest of the file.

In [3]:
csv_text = 'incident_id,service,severity,resolution_hours,deploy_version,customer_impact\nINC-001,api,low,2,v1,no\nINC-002,web,medium,5,v1,yes\nINC-003,worker,high,8,v1,no\nINC-004,api,low,3,v1,yes\nINC-005,web,medium,,v1,no\nINC-006,worker,high,12,v1,yes\nINC-007,api,low,4,v1,no\nINC-008,web,medium,-2,v1,yes\nINC-009,worker,high,9,v1,no\nINC-010,api,low,2,v1,yes\nINC-011,,medium,8,v1,no\nINC-012,worker,high,15,v1,yes\nINC-013,api,low,3,v2,no\nINC-014,web,medium,6,v2,yes\nINC-015,worker,high,10,v2,no\nINC-016,api,low,4,v2,yes\nINC-017,web,medium,,v2,no\nINC-018,worker,high,18,v2,yes\nINC-019,api,low,5,v2,no\nINC-020,web,medium,10,v2,yes\nINC-021,worker,high,14,v2,no\nINC-022,api,low,6,v2,yes\nINC-023,web,medium,11,v2,no\nINC-024,worker,high,24,v2,yes\nINC-003,worker,high,8,v1,no\n'
incidents = pd.read_csv(StringIO(csv_text))
incidents.head()

,incident_id,service,severity,resolution_hours,deploy_version,customer_impact
0,INC-001,api,low,2.0,v1,no
1,INC-002,web,medium,5.0,v1,yes
2,INC-003,worker,high,8.0,v1,no
3,INC-004,api,low,3.0,v1,yes
4,INC-005,web,medium,NaN,v1,no


In [4]:
print("Rows and columns:", incidents.shape)
display(incidents.dtypes)
incidents.info()

Rows and columns: (25, 6)


incident_id             str
service                 str
severity                str
resolution_hours    float64
deploy_version          str
customer_impact         str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   incident_id       25 non-null     str    
 1   service           24 non-null     str    
 2   severity          25 non-null     str    
 3   resolution_hours  23 non-null     float64
 4   deploy_version    25 non-null     str    
 5   customer_impact   25 non-null     str    
dtypes: float64(1), str(5)
memory usage: 1.3 KB


`shape` describes the file. `info()` reports storage and nonmissing counts; it does not validate the values. pandas versions may show text as `str` or `object`; neither determines the statistical type. **Interpret:** Why might the number of rows differ from the number of distinct units?

In [5]:
duration = incidents["resolution_hours"]
duration.head()

0    2.0
1    5.0
2    8.0
3    3.0
4    NaN
Name: resolution_hours, dtype: float64

A selected column is a Series. Missing values may appear as NaN. **Interpret:** Does an unknown duration mean zero hours? Explain in words before continuing.

## S01-E1

Classify incident_id, severity, resolution_hours, and deploy_version. Identify the intended unit and target population. Explain the distinction between a parameter and a statistic in this setting.

In [6]:
classification = None
# TODO: write your classifications in the response cell.

**Your response:**

_Write your explanation here._

## S01-E2

Display the last three records using tail(3). Report the dimensions and storage type of resolution_hours. Identify one concern visible in the records and one that requires checking more than head().

In [7]:
last_records = None
# TODO: use incidents.tail(3) and inspect dimensions/types.

**Your response:**

_Write your explanation here._

## S01-E3

Write a five-sentence audit: unit and scope; dimensions; two quality concerns with evidence; one limitation on a version comparison. Do not clean the data yet.

In [8]:
audit = ""
# TODO: write your audit in the response cell.

**Your response:**

_Write your explanation here._

## Closing synthesis
Before comparing values, establish what a row represents and whether the export
is usable. Submit E1–E3 and complete assessment Q1–Q4. Next session: document
cleaning decisions while preserving the raw export.

## Readings
- [Learning Statistics with Python, 2.1–2.2](https://ethanweed.github.io/pythonbook/01.02-studydesign.html)
- [Think Stats, Chapter 1: DataFrames and Series](https://allendowney.github.io/ThinkStats/chap01.html)
- [pandas read_csv reference](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)